In [2]:
from nvtabular import *
from merlin.schema.tags import Tags
import polars as pl
import xgboost as xgb
from merlin.core.utils import Distributed
from merlin.models.xgb import XGBoost
from nvtabular.ops import AddTags
from torch.onnx.symbolic_opset9 import new_empty

/home/mingyu/miniconda3/envs/xgboost/lib/python3.8/site-packages/merlin/dtypes/mappings/tf.py:52: UserWarning: Tensorflow dtype mappings did not load successfully due to an error: No module named 'tensorflow'
  warn(f"Tensorflow dtype mappings did not load successfully due to an error: {exc.msg}")
/home/mingyu/miniconda3/envs/xgboost/lib/python3.8/site-packages/merlin/dtypes/mappings/triton.py:53: UserWarning: Triton dtype mappings did not load successfully due to an error: No module named 'tritonclient'
  warn(f"Triton dtype mappings did not load successfully due to an error: {exc.msg}")


In [3]:
train=pl.read_parquet('../data/train_and_validation/test.parquet')
train_labels=pl.read_parquet('../data/train_and_validation/test_labels.parquet')

In [4]:
train.head()

session,aid,ts,type
i32,i32,i32,u8
11098528,11830,1661119200,0
11098529,1105029,1661119200,0
11098530,264500,1661119200,0
11098530,264500,1661119288,0
11098530,409236,1661119369,0


In [5]:
train_labels.head()

session,type,ground_truth
i64,str,list[i64]
11098528,"""clicks""",[1679529]
11098528,"""carts""",[1199737]
11098528,"""orders""","[990658, 950341, … 1033148]"
11098529,"""clicks""",[1105029]
11098530,"""orders""",[409236]


In [6]:
# with_column在不修改元DataFrame的前提下，新增或替换一列，返回新的DataFrame
# over相当group_by，需要先定义算什么，再定义在哪算，因此在统计函数之后调用

# 为session赋予按照时间倒序的编号(从0开始)
def add_action_num_reverse_chrono(df):
    return df.select([
        pl.col('*'),
        pl.col('session').cum_count().over('session').alias('action_num_reverse_chrono'),
    ])

# 计算session内的行为总数
def add_session_length(df):
    return df.select([
        pl.col('*'),
        pl.col('session').count().over('session').alias('session_length'),
    ])

# 求解时间位置权重，时间越近，权重越大，将[1,L]映射到[0.1,1],公式为y=y_min+(y_max-y_min)/(x_max-x_min)*pos
def add_lo_recency_score(df):
    return df.with_columns(
        pl.when(pl.col("session_length") == 1)
          .then(1.0)
          .otherwise(
              2 ** (
                  0.1
                  + (1 - 0.1)
                    / (pl.col("session_length") - 1)
                    * (
                        pl.col("action_num_reverse_chrono")
                        - 1
                    )
              ) - 1
          )
          .alias("log_recency_score")
    )


# 按照操作赋予权重
def add_type_weighted_log_recency_score(df):
    type_weight={0:1,1:6,2:3}
    type_weighted_log_recency_score=pl.Series(df['type'].replace(type_weight)*df['log_recency_score'])
    return df.with_columns(type_weighted_log_recency_score.alias('type_weighted_log_recency_score').alias('type_weighted_log_recency_score'))

def apply(df,pipeline):
    for f in pipeline:
        df=f(df)
    return df

In [7]:
pipeline=[add_action_num_reverse_chrono,add_session_length,add_lo_recency_score,add_type_weighted_log_recency_score]
train=apply(train,pipeline)
type2id={'clicks':0,'carts':1,'orders':2}

In [8]:
train.head()

session,aid,ts,type,action_num_reverse_chrono,session_length,log_recency_score,type_weighted_log_recency_score
i32,i32,i32,u8,u32,u32,f64,f64
11098528,11830,1661119200,0,1,1,1.0,1.0
11098529,1105029,1661119200,0,1,1,1.0,1.0
11098530,264500,1661119200,0,1,6,0.071773,0.071773
11098530,264500,1661119288,0,2,6,0.214195,0.214195
11098530,409236,1661119369,0,3,6,0.375542,0.375542


In [9]:
# explode把列表拆成多行
train_labels=train_labels.explode('ground_truth').with_columns([
    pl.col('ground_truth').alias('aid'),
    pl.col('type').replace(type2id)
])[['session','type','aid']]

train_labels=train_labels.with_columns([
    pl.col('session').cast(pl.datatypes.Int32),
    pl.col('type').cast(pl.datatypes.UInt8),
    pl.col('aid').cast(pl.datatypes.Int32),
])

# lit创建一个常量，gt定义正样本
train_labels=train_labels.with_columns(pl.lit(1).alias('gt'))

train=train.join(train_labels,how='left',on=['session','type','aid']).with_columns(pl.col('gt').fill_null(0))

In [10]:
train.head()

session,aid,ts,type,action_num_reverse_chrono,session_length,log_recency_score,type_weighted_log_recency_score,gt
i32,i32,i32,u8,u32,u32,f64,f64,i32
11098528,11830,1661119200,0,1,1,1.0,1.0,0
11098529,1105029,1661119200,0,1,1,1.0,1.0,1
11098530,264500,1661119200,0,1,6,0.071773,0.071773,0
11098530,264500,1661119288,0,2,6,0.214195,0.214195,0
11098530,409236,1661119369,0,3,6,0.375542,0.375542,0


In [11]:
def get_session_lengths(df):
    return df.group_by('session').agg([
        pl.col('session').count().alias('session_length')
    ])['session_length'].to_numpy()

In [12]:
session_lengths_train=get_session_lengths(train)
session_lengths_train

array([ 1,  1,  5, ...,  1, 10,  1], dtype=uint32)

In [13]:
from lightgbm.sklearn import LGBMRanker

In [14]:
ranker=LGBMRanker(
    objective='lambdarank', # 学习的目标，排序用的是lambdarank
    metric='ndcg', # 如何评估模型
    boosting_type='dart', # 如何集成树，dart是带dropout的gbdt
    n_estimators=20, # 树的数量
    importance_type='gain', # 特征重要性
)

In [15]:
train.columns

['session',
 'aid',
 'ts',
 'type',
 'action_num_reverse_chrono',
 'session_length',
 'log_recency_score',
 'type_weighted_log_recency_score',
 'gt']

In [16]:
feature_cols = ['aid', 'type', 'action_num_reverse_chrono', 'session_length', 'log_recency_score', 'type_weighted_log_recency_score']
target = 'gt'

In [17]:
ranker=ranker.fit(train[feature_cols].to_pandas(),train[target].to_pandas(),group=session_lengths_train)# 按照session分组

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024185 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1263
[LightGBM] [Info] Number of data points in the train set: 7683577, number of used features: 6


In [18]:
test=pl.read_parquet('../data/processData/test.parquet')
test=apply(test,pipeline)

In [19]:
score=ranker.predict(test[feature_cols].to_pandas())

In [20]:
test=test.with_columns(pl.Series(name='score', values=score)).sort(['session', 'score'], descending=[False, True])
test

session,aid,ts,type,action_num_reverse_chrono,session_length,log_recency_score,type_weighted_log_recency_score,score
i32,i32,i32,u8,u32,u32,f64,f64,f64
12899779,59625,1661724000,0,1,1,1.0,1.0,0.24219
12899780,736515,1661724136,0,4,5,0.71119,0.71119,0.131451
12899780,1142000,1661724155,0,5,5,1.0,1.0,0.030658
12899780,582732,1661724058,0,2,5,0.252664,0.252664,-0.183435
12899780,973453,1661724109,0,3,5,0.464086,0.464086,-0.21873
…,…,…,…,…,…,…,…,…
14571577,1141710,1662328774,0,1,1,1.0,1.0,0.23439
14571578,519105,1662328775,0,1,1,1.0,1.0,0.243712
14571579,739876,1662328775,0,1,1,1.0,1.0,0.23439


In [21]:
test_predictions = test.group_by('session').agg(
    pl.col('aid').head(20).alias('top20_aid')
)

In [22]:
test_predictions.head()

session,top20_aid
i32,list[i32]
12899779,[59625]
12899780,"[736515, 1142000, … 1142000]"
12899781,"[199008, 918667, … 141736]"
12899782,"[595994, 975116, … 479970]"
12899783,"[607638, 1817895, … 255297]"


In [23]:
session_types=[]
labels=[]

for session,preds in zip(test_predictions['session'].to_numpy(),test_predictions['top20_aid'].to_numpy()):
    l=' '.join(str(p) for p in preds)
    for session_type in ['clicks','carts','orders']:
        labels.append(l)
        session_types.append(f'{session}_{session_type}')

In [24]:
submission=pl.DataFrame({'session_type':session_types,'label':labels})
submission.write_csv('../save/lgb_submission.csv')